# Task
Malware binary classification using the original gated MalConv architecture from [MalTorch](https://github.com/zangobot/maltorch).

Train from scratch on raw executable bytes. No malware is executed and no pretrained weights are downloaded.

---


# Prerequisites
Use Python 3.9+ and a dedicated environment. From this notebook's directory:

```sh
pip-compile requirements.in
pip install -r requirements.txt
```

Select that environment as the notebook kernel. The complete run can be slow on CPU. Files are loaded lazily from the ZIP, rather than keeping all executable bytes or padded inputs in memory. Start with a smaller batch size if GPU memory is limited.


In [ ]:
from datetime import datetime
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path, PurePosixPath
import csv
import io
import random
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn import metrics
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from maltorch.zoo.malconv import MalConv
from tqdm.auto import tqdm

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/backend').is_dir())
NOTEBOOK_DIR = ROOT / 'Training/Malware_Detection'
local_zip = NOTEBOOK_DIR / '.data/MalwareDetectionDataset.zip'
DATA_PATH = local_zip if local_zip.exists() else ROOT / 'src/backend/.data/malware_detection/MalwareDetectionDataset.zip'
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print(f'MalTorch: {version("maltorch")} | PyTorch: {torch.__version__}')
print(f'Dataset: {DATA_PATH}')


---
# Dataset
[DikeDataset](https://github.com/iosifache/DikeDataset): benign = 0, malware = 1. Use the service ZIP with `files/benign`, `files/malware`, `labels/benign`, and `labels/malware` at its root. Label CSVs require `hash`; other label columns are ignored. Class comes from folder membership.

### **Load** the manifest and validate file hashes.
Read samples as data inside the ZIP; do not extract or execute them. Record sizes and PE signatures for the next cells.


In [ ]:
def is_pe_header(prefix, stream, size):
    if len(prefix) < 64 or prefix[:2] != b'MZ':
        return False
    offset = int.from_bytes(prefix[60:64], 'little')
    if offset < 64 or offset + 24 > size:
        return False
    stream.seek(offset)
    return stream.read(4) == b'PE\x00\x00'


# Fingerprint the archive in bounded chunks (works on Python 3.9 too).
archive_hash = sha256()
with DATA_PATH.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        archive_hash.update(chunk)
dataset_version = f'sha256:{archive_hash.hexdigest()}'

rows, label_rows, directories, seen_paths = [], [], set(), set()
required_directories = {'files/benign', 'files/malware', 'labels/benign', 'labels/malware'}
with zipfile.ZipFile(DATA_PATH) as archive:
    for member in tqdm(archive.infolist(), desc='Validate dataset'):
        path = member.filename.rstrip('/')
        parts = path.split('/')
        if path in seen_paths or any(p in ('', '.', '..') for p in parts) or '\\' in path:
            raise ValueError(f'Duplicate or unsafe archive path: {path}')
        seen_paths.add(path)
        if member.is_dir():
            if path not in required_directories | {'files', 'labels'}:
                raise ValueError(f'Unexpected directory: {path}')
            if path in required_directories:
                directories.add(path)
            continue
        if len(parts) != 3 or '/'.join(parts[:2]) not in required_directories:
            raise ValueError(f'Unexpected archive member: {path}')
        directories.add('/'.join(parts[:2]))
        label = int(parts[1] == 'malware')
        if parts[0] == 'labels':
            if PurePosixPath(path).suffix.lower() != '.csv':
                raise ValueError(f'Expected label CSV: {path}')
            with archive.open(member) as stream:
                reader = csv.DictReader(io.TextIOWrapper(stream, encoding='utf-8-sig'))
                if not reader.fieldnames or reader.fieldnames.count('hash') != 1:
                    raise ValueError('Expected exactly one hash column')
                label_rows.extend({'hash': row['hash'], 'label': label} for row in reader)
            continue
        digest = PurePosixPath(path).stem
        if len(digest) != 64 or any(c not in '0123456789abcdef' for c in digest):
            raise ValueError(f'Expected SHA-256 filename: {path}')
        file_hash = sha256()
        with archive.open(member) as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                file_hash.update(chunk)
        if file_hash.hexdigest() != digest or member.file_size == 0:
            raise ValueError(f'Empty file or hash mismatch: {path}')
        with archive.open(member) as stream:
            prefix = stream.read(64)
            pe = is_pe_header(prefix, stream, member.file_size)
        rows.append({'row_id': len(rows), 'hash': digest, 'member': path,
                     'label': label, 'size_bytes': member.file_size, 'is_pe': pe})

if directories != required_directories:
    raise ValueError('All four dataset directories are required')
data = pd.DataFrame(rows)
labels = pd.DataFrame(label_rows)
if data.empty or labels.empty or data['hash'].duplicated().any() or labels['hash'].duplicated().any():
    raise ValueError('Expected nonempty samples and unique hash references')
if set(zip(data['hash'], data['label'])) != set(zip(labels['hash'], labels['label'])):
    raise ValueError('Files and labels must match exactly within each class')
print(f'Loaded {len(data):,} file records | {dataset_version}')


In [ ]:
# File-size visibility immediately after loading. No padded inputs are allocated.
def human_bytes(value):
    value = float(value)
    for unit in ('B', 'KiB', 'MiB', 'GiB', 'TiB'):
        if value < 1024 or unit == 'TiB':
            return f'{value:,.2f} {unit}'
        value /= 1024

size_summary = data.assign(class_name=data['label'].map({0: 'Benign', 1: 'Malware'})).groupby('class_name')['size_bytes'].agg(
    files='count', total='sum', minimum='min', median='median', mean='mean', maximum='max',
)
size_summary.loc['All files'] = [len(data), data['size_bytes'].sum(), data['size_bytes'].min(),
                                 data['size_bytes'].median(), data['size_bytes'].mean(), data['size_bytes'].max()]
display(size_summary.style.format({'files': '{:,.0f}', **{c: human_bytes for c in size_summary.columns if c != 'files'}})
        .set_caption('Executable/archive sample sizes before PE filtering'))
percentiles = data['size_bytes'].quantile([0.5, 0.9, 0.95, 0.99]).rename('Size').to_frame()
percentiles.index = ['P50', 'P90', 'P95', 'P99']
display(percentiles.style.format(human_bytes).set_caption('File-size percentiles'))
display(data.nlargest(10, 'size_bytes')[['hash', 'label', 'size_bytes', 'is_pe']]
        .style.format({'size_bytes': human_bytes}).set_caption('Ten largest files'))
print(f'ZIP size: {human_bytes(DATA_PATH.stat().st_size)} | Total file bytes: {human_bytes(data["size_bytes"].sum())}')


### **Clean** the dataset.
Keep files with DOS/PE signatures, matching the backend's PE screening. Office documents and invalid PE headers are excluded. Hash uniqueness was enforced during loading; original IDs and byte order are preserved.


In [ ]:
data_dropped = data.loc[data['is_pe']].copy().reset_index(drop=True)
if set(data_dropped['label']) != {0, 1}:
    raise ValueError('Training requires both benign and malicious PE samples')
print(f'Retained {len(data_dropped):,} PE samples; removed {len(data) - len(data_dropped):,} non-PE samples.')
display(data_dropped.groupby('label').agg(samples=('hash', 'count'), bytes=('size_bytes', 'sum'))
        .rename(index={0: 'Benign', 1: 'Malware'}).style.format({'samples': '{:,}', 'bytes': human_bytes}))


### Split into Train, Test, and Eval.
Stratified 70/15/15, seed 123, matching the other notebooks. **Test is the validation split** for early stopping. Eval remains untouched until final evaluation. Exact hashes cannot cross splits; related variants may still create optimistic estimates.


In [ ]:
train, holdout = train_test_split(data_dropped, train_size=0.7, random_state=seed, stratify=data_dropped['label'])
test, eval_data = train_test_split(holdout, train_size=0.5, random_state=seed, stratify=holdout['label'])
splits = {'train': train, 'test': test, 'eval': eval_data}
split_hashes = {part: frame['hash'].tolist() for part, frame in splits.items()}
assert len(set(sum(split_hashes.values(), []))) == len(data_dropped)
display(pd.DataFrame({part: {'samples': len(frame), 'benign': int((frame['label'] == 0).sum()),
                            'malware': int((frame['label'] == 1).sum())} for part, frame in splits.items()}).T)


---
# Preprocessing
Preserve raw bytes 0–255; pad with 256. Read only the leading `max_length` bytes for each batch. Longer files are truncated, so predictions and explanations cover only that prefix. Hashes and row IDs are tracking metadata, never model inputs.


In [ ]:
model_config = dict(embedding_size=8, max_len=2**20, threshold=0.5,
                    padding_idx=256, kernel_size=512, stride=512)
max_length = model_config['max_len']
if max_length < model_config['kernel_size']:
    raise ValueError('Input length must accommodate the convolution kernel')
coverage = pd.DataFrame({part: {'files': len(frame), 'truncated': int((frame['size_bytes'] > max_length).sum()),
                               'truncated_percent': 100 * (frame['size_bytes'] > max_length).mean()}
                         for part, frame in splits.items()}).T
display(coverage.style.format({'files': '{:,.0f}', 'truncated': '{:,.0f}', 'truncated_percent': '{:.2f}%'}))
print(f'Input limit: {human_bytes(max_length)} | Byte offset: 0 | Padding: 256')


class ExecutableBytesDataset(Dataset):
    def __init__(self, archive_path, frame, max_length, padding_token=256):
        self.archive_path = str(archive_path)
        self.frame = frame.reset_index(drop=True).copy()
        self.max_length = max_length
        self.padding_token = padding_token
        self._archive = None

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        if self._archive is None:
            self._archive = zipfile.ZipFile(self.archive_path)
        row = self.frame.iloc[index]
        with self._archive.open(row['member']) as stream:
            prefix = stream.read(self.max_length)
        tokens = torch.full((self.max_length,), self.padding_token, dtype=torch.long)
        tokens[:len(prefix)] = torch.from_numpy(np.frombuffer(prefix, dtype=np.uint8).copy()).long()
        return tokens, torch.tensor(int(row['label']), dtype=torch.float32)

    def close(self):
        if self._archive is not None:
            self._archive.close()
            self._archive = None

    def __getstate__(self):
        state = self.__dict__.copy()
        state['_archive'] = None
        return state


---
# PyTorch settings


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
batch_size = 4
loader_generator = torch.Generator().manual_seed(seed)
byte_datasets = {part: ExecutableBytesDataset(DATA_PATH, frame, max_length) for part, frame in splits.items()}
# num_workers=0 works inside notebooks across platforms and avoids ZIP handle sharing.
loaders = {part: DataLoader(dataset, batch_size=batch_size, shuffle=(part == 'train'),
                           num_workers=0, pin_memory=(device.type == 'cuda'),
                           generator=loader_generator if part == 'train' else None)
           for part, dataset in byte_datasets.items()}
print(f'Device: {device} | Batch size: {batch_size} | Input length: {max_length:,}')


---
# Model and Optimizer
Use MalTorch's trainable `MalConv(...)` module, not `create_model()`, which loads a pretrained wrapper. Its output is a single logit per sample; use BCEWithLogitsLoss and apply sigmoid only for probabilities.


In [ ]:
torch.manual_seed(seed)
model = MalConv(**model_config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()
gradient_clipping_norm = 1.0
print(model)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


---
# Training
Monitor test/validation loss for early stopping. Save every epoch and a separate `best.pt`; eval does not participate in checkpoint selection.

### Helper functions.


In [ ]:
@torch.inference_mode()
def predict_probabilities(net, loader):
    net.eval()
    probabilities, labels = [], []
    for tokens, targets in loader:
        logits = net(tokens.to(device, non_blocking=True)).reshape(-1)
        probabilities.append(logits.sigmoid().cpu())
        labels.append(targets)
    return torch.cat(labels).numpy().astype(np.int64), torch.cat(probabilities).numpy()


def checkpoint_values(value):
    # Keep NumPy metric scalars compatible with weights_only=True loading.
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {key: checkpoint_values(item) for key, item in value.items()}
    if isinstance(value, list):
        return [checkpoint_values(item) for item in value]
    if isinstance(value, tuple):
        return tuple(checkpoint_values(item) for item in value)
    return value


def save_checkpoint(payload, path):
    temporary = path.with_suffix('.tmp')
    torch.save(checkpoint_values(payload), temporary)
    temporary.replace(path)


### Training loop.
Run this cell when ready to train. Reduce `batch_size` above if memory is insufficient. The default run may take substantial time on CPU.


In [ ]:
n_epochs = 100
patience = 16
best_test_loss = float('inf')
best_epoch = None
epochs_without_improvement = 0
training_history = []
checkpoint_dir = NOTEBOOK_DIR / '.models' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
checkpoint_dir.mkdir(parents=True, exist_ok=False)
print(f'Checkpoint folder: {checkpoint_dir}')

for epoch in range(1, n_epochs + 1):
    model.train()
    total_loss = 0.0
    for tokens, targets in tqdm(loaders['train'], desc=f'Epoch {epoch}', leave=False):
        tokens, targets = tokens.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(tokens).reshape(-1)
        loss = loss_fn(logits, targets)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping_norm)
        optimizer.step()
        total_loss += loss.item() * len(targets)

    test_labels, test_probabilities = predict_probabilities(model, loaders['test'])
    test_loss = metrics.log_loss(test_labels, test_probabilities, labels=[0, 1])
    training_history.append(dict(epoch=epoch, train_loss=total_loss / len(train), test_loss=test_loss,
                                 test_accuracy=metrics.accuracy_score(test_labels, test_probabilities >= 0.5),
                                 test_pr_auc=metrics.average_precision_score(test_labels, test_probabilities)))
    improved = test_loss < best_test_loss
    if improved:
        best_test_loss, best_epoch, epochs_without_improvement = test_loss, epoch, 0
    else:
        epochs_without_improvement += 1

    checkpoint = {
        'epoch': epoch, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
        'model_config': model_config.copy(),
        'preprocessing': {'max_length': max_length, 'byte_offset': 0, 'padding_token': 256},
        'model_library': {'name': 'maltorch', 'version': version('maltorch'), 'class': 'maltorch.zoo.malconv.MalConv'},
        'torch_version': str(torch.__version__), 'label_map': {'benign': 0, 'malware': 1},
        'dataset_version': dataset_version, 'split_hashes': split_hashes,
        'split_row_ids': {part: frame['row_id'].tolist() for part, frame in splits.items()},
        'metrics': training_history[-1].copy(), 'training_history': training_history.copy(),
        'best_epoch': best_epoch, 'best_test_loss': best_test_loss,
        'epochs_without_improvement': epochs_without_improvement,
        'training_config': {'seed': seed, 'batch_size': batch_size, 'patience': patience,
                            'n_epochs': n_epochs, 'gradient_clipping_norm': gradient_clipping_norm},
        'torch_rng_state': torch.get_rng_state(), 'python_rng_state': random.getstate(),
        'numpy_rng_state': [np.random.get_state()[0], np.random.get_state()[1].tolist(),
                            *np.random.get_state()[2:]],
        'loader_rng_state': loader_generator.get_state(),
        'cuda_rng_state': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }
    save_checkpoint(checkpoint, checkpoint_dir / f'epoch_{epoch:03d}.pt')
    if improved:
        save_checkpoint(checkpoint, checkpoint_dir / 'best.pt')
    print(f'Epoch {epoch:3} | Train loss: {training_history[-1]["train_loss"]:.4f} | '
          f'Test loss: {test_loss:.4f} | Test accuracy: {training_history[-1]["test_accuracy"]:.4f}')
    if epochs_without_improvement >= patience:
        print(f'Early stopping after {patience} epochs without improvement.')
        break

history_df = pd.DataFrame(training_history)
print(f'Best epoch: {best_epoch} | Best test loss: {best_test_loss:.4f}')
print(f'Best checkpoint: {checkpoint_dir / "best.pt"}')


### Plot training progress.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
history_df.plot(x='epoch', y=['train_loss', 'test_loss'], ax=axes[0], title='Loss')
history_df.plot(x='epoch', y='test_accuracy', ax=axes[1], title='Test accuracy')
history_df.plot(x='epoch', y='test_pr_auc', ax=axes[2], title='Test average precision (PR-AUC summary)')
for ax in axes:
    ax.axvline(best_epoch, color='gray', linestyle='--', label='Best validation loss')
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


---
# Evaluation
### Load the best checkpointed model.
`best.pt` is selected by validation loss automatically. Use a saved run path if evaluating in a later session. Only load checkpoints from a trusted run.


In [ ]:
loaded_checkpoint_path = checkpoint_dir / 'best.pt'
loaded_checkpoint = torch.load(loaded_checkpoint_path, map_location='cpu', weights_only=True)
if loaded_checkpoint['dataset_version'] != dataset_version:
    raise ValueError('Checkpoint belongs to a different dataset archive')
if loaded_checkpoint['model_library']['version'] != version('maltorch'):
    raise ValueError('Use the MalTorch version saved in the checkpoint')
model = MalConv(**loaded_checkpoint['model_config']).to(device)
model.load_state_dict(loaded_checkpoint['model'])
model.eval()
# Restore exact eval membership and preprocessing from the saved run.
by_hash = data_dropped.set_index('hash', drop=False)
loaded_eval = by_hash.loc[loaded_checkpoint['split_hashes']['eval']].reset_index(drop=True)
preprocessing = loaded_checkpoint['preprocessing']
eval_dataset = ExecutableBytesDataset(DATA_PATH, loaded_eval, preprocessing['max_length'], preprocessing['padding_token'])
eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
print(f'Loaded epoch {loaded_checkpoint["epoch"]}: {loaded_checkpoint_path}')
print(f'Saved validation metrics: {loaded_checkpoint["metrics"]}')


### Evaluate the held-out Eval split.
The decision threshold is fixed at 0.5. If choosing another threshold, use the validation split before running this cell.


In [ ]:
eval_labels, eval_probabilities = predict_probabilities(model, eval_loader)
eval_predictions = (eval_probabilities >= loaded_checkpoint['model_config']['threshold']).astype(int)
eval_metrics = pd.Series({
    'log_loss': metrics.log_loss(eval_labels, eval_probabilities, labels=[0, 1]),
    'accuracy': metrics.accuracy_score(eval_labels, eval_predictions),
    'precision': metrics.precision_score(eval_labels, eval_predictions, zero_division=0),
    'recall': metrics.recall_score(eval_labels, eval_predictions, zero_division=0),
    'f1': metrics.f1_score(eval_labels, eval_predictions, zero_division=0),
    'roc_auc': metrics.roc_auc_score(eval_labels, eval_probabilities),
    'average_precision': metrics.average_precision_score(eval_labels, eval_probabilities),
})
display(eval_metrics.to_frame('Eval').style.format('{:.4f}'))
print(metrics.classification_report(eval_labels, eval_predictions, labels=[0, 1],
                                    target_names=['Benign', 'Malware'], zero_division=0))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
metrics.ConfusionMatrixDisplay.from_predictions(eval_labels, eval_predictions, display_labels=['Benign', 'Malware'], ax=axes[0])
metrics.RocCurveDisplay.from_predictions(eval_labels, eval_probabilities, ax=axes[1])
metrics.PrecisionRecallDisplay.from_predictions(eval_labels, eval_probabilities, ax=axes[2])
fig.tight_layout()
plt.show()
pd.DataFrame({'hash': loaded_eval['hash'], 'label': eval_labels, 'malware_probability': eval_probabilities,
              'prediction': eval_predictions}).to_csv(checkpoint_dir / 'eval_predictions.csv', index=False)
eval_metrics.to_json(checkpoint_dir / 'eval_metrics.json', indent=2)


### Close dataset handles.
Run after evaluation (or before rerunning preprocessing cells). No sample files have been extracted.


In [ ]:
for dataset in byte_datasets.values():
    dataset.close()
eval_dataset.close()
